## Connecting to GPU

In [ ]:
gpu_info = !nvidia-smi
gpu_info

## Installing Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, AutoProcessor
import torch

In [ ]:
cache_dir = "./Falcon3-1B-Instruct"

In [ ]:
model_name = "tiiuae/Falcon3-1B-Instruct"

### Quantization Config

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load the model in 4-bit precision
    bnb_4bit_use_double_quant=True,  # Use double quantization for better accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for computation
    bnb_4bit_quant_type="nf4",  # Use NormalFloat4 quantization
)

quant_config

### Tokenizer

In [ ]:
# Download and cache tokenizer & config
# tokenizer = AutoTokenizer.from_pretrained(
#     model_name,
#     cache_dir=cache_dir,
#     use_fast=True
# )

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     cache_dir=cache_dir,
#     quantization_config=quant_config,
#     device_map="auto"
# )

In [ ]:
transcript = "Hello, how are you? We are planning to conduct our Annual next meeting on 24th February 2026 at 4 pm at Main Auditorium."

In [19]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date if available
- discussion points if available
- takeaways if available
- action items with owners if available

Do not add anyting that not in the context

Transcription:
{transcript}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [20]:
messages

[{'role': 'system',
  'content': '\nYou produce minutes of meetings from transcripts, with summary, key discussion points,\ntakeaways and action items with owners, in markdown format without code blocks.\n'},
 {'role': 'user',
  'content': '\nBelow is an extract transcript of a meeting.\nPlease write minutes in markdown without code blocks, including:\n- a summary with attendees, location and date if available\n- discussion points if available\n- takeaways if available\n- action items with owners if available\n\nDo not add anyting that not in the context\n\nTranscription:\nHello, how are you? We are planning to conduct our Annual next meeting on 24th February 2026 at 4 pm at Main Auditorium.\n'}]

In [21]:
tokenizer = AutoProcessor.from_pretrained(
    "./Falcon3-1B-Instruct",  # root repo folder
    local_files_only=True,     # no download
)
tokenizer.pad_token = tokenizer.eos_token 

In [22]:
input_ids = tokenizer.apply_chat_template(
    messages, 
    return_tensors="pt", 
    padding=True, 
    truncation=True).to("cuda")
    
attention_mask = torch.ones_like(input_ids, dtype=torch.long).to("cuda")
streamer = TextStreamer(tokenizer)

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    "./Falcon3-1B-Instruct",
    device_map="cuda:0",           
    dtype=torch.float16,
    max_memory={0: "4GiB"},   
    local_files_only=True,
    quantization_config=quant_config,
)

In [44]:
output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


In [51]:
details = tokenizer.decode(output[0], skip_special_tokens=False)

In [55]:
Markdown(details)

<|system|>

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.

<|user|>

Below is an extract transcript of a meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date if available
- discussion points if available
- takeaways if available
- action items with owners if available

Do not add anyting that not in the context

Transcription:
Hello, how are you? We are planning to conduct our Annual next meeting on 24th February 2026 at 4 pm at Main Auditorium.

<|assistant|>
# Annual Meeting Minutes

## Summary
- **Date:** 24th February 2026
- **Location:** Main Auditorium
- **Attendees:** [List of Attendees]

## Discussion Points
- **Agenda Overview:**
  - **Business Review:** [Details of Business Review]
  - **Strategic Planning:** [Details of Strategic Planning]
  - **Financial Review:** 